# Notebook para coleta e tratamento dos dados primários

Utilizarei APIs abertos e coletas manuais para a criação das bases de dados primárias.

1 - Calendario e clima

2 - Artistas

3 - Shows

4 - Locais (manual)

5 - Setores

6 - Preços por setores por show (manual)

7 - Festivais

In [ ]:
import pandas as pd
import requests
import time
from datetime import datetime, timedelta
from dateutil import parser
from sqlalchemy import create_engine
import os
from spotipy import Spotify
from spotipy.oauth2 import SpotifyClientCredentials
import json
import base64
import random
import math
from dotenv import load_dotenv
import string

### Base Calendário

Construída com a estrutura em Python, BrasilAPI (https://brasilapi.com.br/api/feriados/v1/), agenda cultural oficial de São Paulo. Já os dados climáticos são importados do API do Open Meteo (https://archive-api.open-meteo.com/v1/archive).
A base será construída considerando os seguintes filtros:
* Cidade: São Paulo (SP)
* País: Brasil
* Data inicial: 01/01/2021 (ano em que começou a reabertura de eventos após a pandemia)
* Data final: 31/12/2026 (considerando que não há, ainda, shows confirmados para 2027)

Colunas da base: Data, ano, dia da semana, tipo de dia (feriado municipal, feriado nacional, dia útil, fim de semana), nome do feriado, temperatura média, precipitação, umidade, condições climáticas.

In [ ]:
# Filtros

st_date = "2021-01-01"
end_date = "2026-12-31"
city = "São Paulo"
uf = "SP"

In [ ]:
# Calendario de datas
dates = pd.date_range(start = st_date, end = end_date, freq = "D")
dt = pd.DataFrame({"data": dates})
dt["data"] = dt["data"].dt.date
dt["ano"] = pd.DatetimeIndex(dt["data"]).year
dt["dia_semana"] = pd.DatetimeIndex(dt["data"]).day_name(locale = "pt_BR")

In [ ]:
# Inclusão dos feriados nacionais com o BrasilAPI e Municipais manualmente segundo a agenda da cidade
def feriados_nac_func(ano):
    url_fer = f"https://brasilapi.com.br/api/feriados/v1/{ano}"
    resp_fer = requests.get(url_fer, timeout = 30) # timrout = tempo máximo de resposta
    resp_fer.raise_for_status() # Excessão caso haja erro no retorno do link
    return resp_fer.json()

feriados = []
# Feriados Nacionais
for ano in range (dt["ano"].min(), dt["ano"].max() + 1):
    try:
        data_api = feriados_nac_func(ano)
        for item in data_api:
            if isinstance(item, dict) and "date" in item and "name" in item:
                feriados.append((item["date"], item["name"], "Feriado Nacional"))
            else:
                print(f"⚠️ Estrutura inesperada em {ano}: {item}")
    except Exception as e:
        print(f"Erro ao buscar feriados nacionais de {ano}: {e}")

# Feriados Municipais e Estaduais
for ano in range (dt["ano"].min(), dt["ano"].max() + 1):
    # Aniversário da Cidade (25/jan)
    feriados.extend([
        (f"{ano}-01-25", "Aniversário de São Paulo", "Feriado Municipal"),
    # Revolução Constitucionalista (07/Set)
        (f"{ano}-09-07", "Revolução Constitucionalista", "Feriado Estadual")
    ])

    # Consciência Negra (20/Nov até 2023)
for ano in range (dt["ano"].min(), 2024):
    feriados.extend([(f"{ano}-09-07", "Consciência Negra", "Feriado Municipal")])

len(feriados), feriados[0]

df_feriados = pd.DataFrame(feriados, columns = ["data_str", "nome_feriado", "tipo_dia"])
df_feriados["data"] = pd.to_datetime(df_feriados["data_str"]).dt.date
dt["data"] = pd.to_datetime(dt["data"]).dt.date
dt = dt.merge(df_feriados[["data", "nome_feriado", "tipo_dia"]], on="data", how="left")

In [ ]:
# Tratando a coluna tipo de dia (dia útil, fim de semana, feriados já existem)
dt.loc[dt["data"].apply(lambda x: pd.Timestamp(x).weekday()) >= 5, "tipo_dia"] = "Fim de semana"
dt["tipo_dia"] = dt["tipo_dia"].fillna("Dia útil")

In [ ]:
# Adicionar os dados de tempo (API do OpenWeatherMap)
lat, lon = -23.55, -46.6333
clima = []

In [ ]:
# Extrair dados que interessam
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": lat,
    "longitude": lon,
    "start_date": st_date,
    "end_date": pd.Timestamp.today().strftime("%Y-%m-%d"),
    "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum,weathercode",
    "timezone": "America/Sao_Paulo"
}
response = requests.get(url, params=params)
data = response.json()
df_cl = pd.DataFrame(data["daily"])

dt["data"] = pd.to_datetime(dt["data"]).dt.date
df_cl["time"] = pd.to_datetime(df_cl["time"]).dt.date

dt = dt.merge(df_cl, left_on="data", right_on="time", how="left")

In [ ]:
# Traduzir o código sobre a descrição climática
weather_map = {
    0: "Céu limpo", 1: "Predominantemente limpo", 2: "Parcialmente nublado", 3: "Nublado",
    45: "Névoa", 48: "Neblina", 51: "Chuvisco leve", 53: "Chuvisco moderado", 55: "Chuvisco intenso",
    61: "Chuva leve", 63: "Chuva moderada", 65: "Chuva forte",
    80: "Pancadas de chuva leves", 81: "Pancadas de chuva moderadas", 82: "Pancadas de chuva fortes",
    95: "Trovoadas leves", 96: "Trovoadas com granizo", 99: "Trovoadas severas com granizo"
}

# Adicionar descrição textual
dt["descricao_clima"] = dt["weathercode"].map(weather_map)

In [ ]:
dt.info()

In [ ]:
# Ordenar a base
dt.drop(columns=["time"], inplace=True)
dt = dt.sort_values("data").reset_index(drop=True)

dt.head(10)

In [ ]:
# Salvaro CSV da base
out_path = "../data/external/calendario_sp_2021_2025.csv"
dt.to_csv(out_path, index=False, encoding="utf-8-sig")

### Base Shows

Para montagem dessa base de dados utiizarei o API do dite Setlistfm (api.setlist.fm/rest/1.0/search/setlists), com os mesmo filtros utlizados anteriormente na base de calendário e com as colunas:

showid, data, artista, localid, turne, publico estimado

In [ ]:
api_key = "57lRYWJaT56UEXM5DGNjPp9df2ccKw_ab4bZ"

In [ ]:
# Definições
pais = "BR"
city = "São Paulo"
uf = "SP"


In [ ]:
shows = []

for ano in range(2021, 2026):
    pagina = 1
    total_paginas = 0  # Para rastrear

    while pagina <= 150:  # Limite razoável por ano (SP tem ~200–600 shows/ano)
        try:
            url = "https://api.setlist.fm/rest/1.0/search/setlists"
            headers = {
                "x-api-key": api_key,
                "Accept": "application/json"
            }
            params = {
                "cityName": city,
                "countryCode": pais,
                "stateCode": uf,
                "p": pagina,
                "year": ano
            }

            resp = requests.get(url, headers=headers, params=params, timeout=30)

            # Rate limit
            if resp.status_code == 429:
                print(f"⚠️ Rate limit no ano {ano}, página {pagina}. Aguardando 60s...")
                time.sleep(60)
                continue

            # 404 ou vazio = sem mais resultados para esse ano
            if resp.status_code == 404:
                print(f"✅ Ano {ano} concluído na página {pagina} (404 = sem mais dados)")
                break
            elif resp.status_code == 200:
                data = resp.json()
                setlists = data.get("setlist", [])

                if not setlists:  # Lista vazia também para o loop
                    print(f"✅ Ano {ano} concluído na página {pagina} (lista vazia)")
                    break

                print(f"Ano {ano} | Página {pagina}: +{len(setlists)} shows coletados")

                # Processa shows
                for s in setlists:
                    try:
                        data_show = pd.to_datetime(s["eventDate"], format="%d-%m-%Y", errors="coerce")
                        if pd.notna(data_show) and data_show.year == ano:
                            shows.append({
                                "artista": s["artist"]["name"],
                                "data": data_show.date(),
                                "ano": data_show.year,
                                "local": s["venue"]["name"],
                                "cidade": s["venue"]["city"]["name"],
                                "estado": s["venue"]["city"].get("state", "SP")  # Adicionei para completude
                            })
                    except Exception as e:
                        print(f"⚠️ Erro processando show: {e}")
                        continue

                total_paginas = pagina
                pagina += 1
            else:
                print(f"❌ Erro HTTP {resp.status_code} no ano {ano}, página {pagina}: {resp.text[:200]}")
                break

            time.sleep(1.5)  # Pausa para respeitar ~1000 req/hora

        except requests.exceptions.Timeout:
            print(f"⏰ Timeout no ano {ano}, página {pagina}. Tentando de novo...")
            time.sleep(10)
            continue
        except Exception as e:
            print(f"💥 Erro geral no ano {ano}, página {pagina}: {e}")
            time.sleep(5)
            break

    print(f"--- Ano {ano} finalizado: {total_paginas} páginas processadas ---\n")

# DataFrame final
df_shows = pd.DataFrame(shows)
if not df_shows.empty:
    df_shows["data"] = pd.to_datetime(df_shows["data"], errors="coerce")
    df_shows["ano"] = df_shows["data"].dt.year
    df_shows = df_shows[df_shows["ano"].between(2021, 2025)].sort_values("data").reset_index(drop=True)

    print(f"✅ Total de shows coletados em São Paulo: {len(df_shows):,}")
    print(f"Período coberto: {df_shows['data'].min().date()} a {df_shows['data'].max().date()}")
    print("\nTop 10 artistas com mais shows:")
    print(df_shows["artista"].value_counts().head(10))

    display(df_shows.head(10))
else:
    print("❌ Nenhum show coletado. Verifique a API key ou parâmetros.")

In [ ]:
df_shows.to_csv("../data/external/shows.csv", index=False, encoding="utf-8-sig")

### Base Artistas
Construída com a API do Spotify (https://api.spotify.com/v1/artists). Sua estrutura terá as seguintes colunas:

ArtistaID, Artista, Gênero, Popularidade, Seguidores no Spotify

Como essaserá uma base grande com muitas chamadas, utilizarei 4 acessos diferentes ao API.


In [ ]:
CLIENTES = [
    ("f36269a21de14e88a0466040a8d02fa4", "1633ad049c1c49d39ecab78eefe05991"),
    ("e702987d67f54a65bc68b6e9a85be7ae", "477cef370bc546fc8570d2354881e4cb"),
    ("883b8e3992b44210a94f51db306cb55c", "2aac4318c8044bbfa3a4368782d42db2"),
    ("33505d61e70e43bdaf88f24b9b9de826", "f6853c2b2ed2444eabd2222a223311db"),
    ("e00b62afd6fc4c6bbfa5232846649cfe", "0d78ad8a3b2d477396db0566fefeedfd")
]

In [ ]:
# LISTA DEFINITIVA 2025 — PEGA QUASE TODOS OS ARTISTAS RELEVANTES NO BRASIL + INTERNACIONAIS
GENRES = [
    "sertanejo pop",          # Gusttavo Lima, Ana Castela, Zé Neto, etc.
    "sertanejo",              # versão pura — pega muitos também
    "sertanejo universitario",
    "arrocha",
    "country",
    "forro",                  # Wesley Safadão, João Gomes, etc.
    "funk carioca",           # Anitta, Ludmilla, MCs
    "funk mtg",
    "brega funk",             # MC Pipokinha, Conde, etc.
    "pagode",                 # Thiaguinho, Grupo Revelação, etc.
    "pagode baiano",
    "samba",
    "mpb",
    "brazilian",              # pega MPB, bossa, samba-rock, etc.
    "bossa nova",
    "brazilian hip hop",      # Criolo, Emicida, Racionais
    "trap brasileiro",        # Matuê, WIU, Teto, Recayd
    "trap",                   # pega brasileiros + alguns gringos
    "drill brasileiro",
    "pop",                    # Marília, Luísa Sonza, Jão, etc.
    "brazilian pop",
    "pop nacional",
    "rock",                   # CPM22, Fresno, NX Zero, etc.
    "brazilian rock",
    "indie",
    "indie rock",
    "emo",                    # Fresno, NX Zero, Pitty
    "pop punk",
    "punk rock",
    "metal",                  # Sepultura, Angra, etc.
    "brazilian metal",
    "eletronica",
    "brazilian edm",          # Alok, Vintage Culture, Bhaskar
    "bass house",
    "deep house",
    "reggaeton",              # Bad Bunny, Karol G, etc.
    "latin pop",              # Shakira, Ricky Martin, etc.
    "latin",                  # pega muitos que tocam em SP
    "urbano latino",
    "rap",
    "hip hop",
    "trap latino",
    "drill",
    "k-pop",                  # BTS, Blackpink, etc. (muito show em SP)
    "afrobeats",              # Burna Boy, etc.
    "r&b",
    "dance pop"
]

GENRES_1 = GENRES[:len(GENRES)//2]
GENRES_2 = GENRES[len(GENRES)//2:]

LETTERS_1 = list(string.ascii_uppercase)[:13]
LETTERS_2 = list(string.ascii_uppercase)[13:]

NUM = ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']

LIMIT = 50
MAX_PAGES = 10
MIN_POPULARITY = 59
MIN_FOLLOWERS = 150000

PARCIAL = "../data/temp/parcial.json"
CSV_FINAL = "../data/external/artistas.csv"

In [ ]:
def autenticar(indice):
    cid, secret = CLIENTES[indice]
    print(f"\n🔐 Trocando para CLIENTE #{indice+1}: {cid[:10]}...")
    return Spotify(
        auth_manager=SpotifyClientCredentials(
            client_id=cid,
            client_secret=secret
        )
    )

In [ ]:
def load_parcial():
    try:
        with open(PARCIAL, "r", encoding="utf-8") as f:
            todos = json.load(f)
        print(f"🟢 Parcial carregada: {len(todos)} registros")
    except:
        todos = []
        print("⚪ Nenhuma parcial encontrada. Iniciando nova coleta.")
    return todos

def save_parcial(todos):
    with open(PARCIAL, "w", encoding="utf-8") as f:
        json.dump(todos, f, ensure_ascii=False, indent=2)
    print(f"💾 Salvo incremental (total = {len(todos)})")

In [ ]:
def extract_from_item(a, fonte):
    followers = a.get("followers", {}).get("total", 0)
    genres = ", ".join(a.get("genres", []))
    images = a.get("images", [])
    image_url = images[0]["url"] if images else ""
    return {
        "nome": a.get("name", ""),
        "id_spotify": a.get("id", ""),
        "generos": genres,
        "popularidade": a.get("popularity", 0),
        "seguidores": followers,
        "imagem": image_url,
        "fonte": fonte
    }

In [ ]:
def coletar(sp, query_list, fonte_label):
    todos = load_parcial()

    for termo in query_list:
        print(f"\n🎧 Termo: {termo}")
        for pag in range(MAX_PAGES):
            try:
                r = sp.search(
                    q=termo,
                    type='artist',
                    limit=LIMIT,
                    offset=pag * LIMIT
                )
                items = r.get("artists", {}).get("items", [])

                if not items:
                    break

                for a in items:
                    if a.get("popularity", 0) < MIN_POPULARITY:
                        continue
                    seg = a.get("followers", {}).get("total", 0)
                    if seg >= MIN_FOLLOWERS:
                        todos.append(extract_from_item(a, fonte_label))

                    if len(todos) % 50 == 0:
                        save_parcial(todos)

                time.sleep(random.uniform(1.3, 2.4))

            except Exception as e:
                print("⚠️ Erro:", e, "→ aguardando 8s...")
                time.sleep(8)

    save_parcial(todos)
    print("✔ Coleta concluída para esse bloco.")
    return todos

In [ ]:
if __name__ == "__main__":

    # BLOCO 1 – GENRES_1
    sp = autenticar(0)
    coletar(sp, [f'genre:"{g}"' for g in GENRES_1], "GENRE_1")

    # BLOCO 2 – GENRES_2
    sp = autenticar(1)
    coletar(sp, [f'genre:"{g}"' for g in GENRES_2], "GENRE_2")

    # BLOCO 3 – LETRAS_1
    sp = autenticar(2)
    coletar(sp, [f"{l}*" for l in LETTERS_1], "LETTER_1")

    # BLOCO 4 – LETRAS_2
    sp = autenticar(3)
    coletar(sp, [f"{l}*" for l in LETTERS_2[:-1]], "LETTER_2")

    # BLOCO 5 - Números
    sp =autenticar(4)
    coletar(sp, [f"{n}*" for n in NUM], "NUM")

    # =====================================================
    # FINAL — DEDUPLICAÇÃO + CSV
    # =====================================================
    print("\n📦 Gerando CSV...")

    todos = load_parcial()
    df = pd.DataFrame(todos).drop_duplicates(subset="id_spotify")
    df = df.sort_values("seguidores", ascending=False)

    df.to_csv(CSV_FINAL, index=False, encoding="utf-8-sig")
    print(f"🎉 CSV FINAL SALVO! Total de artistas únicos: {len(df)}")
    print("Arquivo:", CSV_FINAL)


🔐 Trocando para CLIENTE #1: f36269a21d...
🟢 Parcial carregada: 43781 registros

🎧 Termo: genre:"sertanejo pop"

🎧 Termo: genre:"sertanejo"

🎧 Termo: genre:"sertanejo universitario"

🎧 Termo: genre:"arrocha"

🎧 Termo: genre:"country"
💾 Salvo incremental (total = 43800)
💾 Salvo incremental (total = 43850)
💾 Salvo incremental (total = 43900)
💾 Salvo incremental (total = 43950)

🎧 Termo: genre:"forro"

🎧 Termo: genre:"funk carioca"

🎧 Termo: genre:"funk mtg"

🎧 Termo: genre:"brega funk"

🎧 Termo: genre:"pagode"

🎧 Termo: genre:"pagode baiano"

🎧 Termo: genre:"samba"

🎧 Termo: genre:"mpb"

🎧 Termo: genre:"brazilian"

🎧 Termo: genre:"bossa nova"

🎧 Termo: genre:"brazilian hip hop"

🎧 Termo: genre:"trap brasileiro"

🎧 Termo: genre:"trap"

🎧 Termo: genre:"drill brasileiro"

🎧 Termo: genre:"pop"
💾 Salvo incremental (total = 44000)
💾 Salvo incremental (total = 44050)
💾 Salvo incremental (total = 44100)
💾 Salvo incremental (total = 44150)
💾 Salvo incremental (total = 44200)
💾 Salvo incremental (

### Base de Locais
Em relação a base de locias onde os eventos são realizados, os dados considerados serão os presente no item venue id do setlistfm com dados coletados manualmente. Locais com capacidade superior à 2000 pessoas. E com as colunas:

localid, nome do local, bairro, capacidade total, tipo de local, latitude e longitude.

###### Base: locais.csv

### Base de setores

Essa base tem como objetivo detalhar os possíveis setores dos eventos. Com dados coletados manualmente nos sites de venda de ingeresso e sites de informações de shows.

setorid, nome do setor, categoria setor

###### Base: setores.csv

## Base de valores de ingressos

Base construída com pesquisa manual, considerando a base de dados de shows previamente tratada e filtrada apenas com os shows de interesse para o projeto.

artista | data | setor | valor

##### Base: valores.cvs

## Base de festivais

Base construída com pesquisa manual, aprensentando os maiores festivais musicias que ocoreram no período em São Paulo.

nome | local | data | ano | headliners

##### Base: festivais.csv